# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nancy121205/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Baseline rule:** Prioritize pages with at least 500 impressions in the last 90 days that show one or more review signals: a search position between 11 and 30, an update age of at least 180 days, or a CTR below the median for comparable visible pages.

The score is transparent: 3 points for being in striking distance (position 11–30), 2 points for stale content, and 1 point for below-median CTR. Pages below the visibility threshold receive a score of zero.

Reason codes explain which signals triggered the score. This is a starting heuristic, not a trained model or proof that a page will improve after editing.


In [1]:

# Section 1: Rule definition

MIN_IMPRESSIONS = 500
STRIKING_MIN_POSITION = 11
STRIKING_MAX_POSITION = 30
STALE_DAYS = 180

WEIGHT_STRIKING = 3
WEIGHT_STALE = 2
WEIGHT_LOW_CTR = 1

REASON_CODES = {
    "striking_distance": "Average position is between 11 and 30",
    "stale_content": "Last update was at least 180 days ago",
    "low_ctr": "CTR is below the comparison-group median",
    "low_visibility": "Fewer than 500 impressions; score is zero",
    "no_rule_trigger": "No positive review signal triggered",
}

print("Rule configured.")
print("Reason codes:", sorted(REASON_CODES))

Rule configured.
Reason codes: ['low_ctr', 'low_visibility', 'no_rule_trigger', 'stale_content', 'striking_distance']


## 2. Build the ranked queue (writes the CSV)

I will calculate the score using the rule above, rank every content item, and preserve reason codes alongside the score. The CSV will include the pseudonymous content ID for traceability, but IDs will not affect the score.

The ranking is deterministic: identical input data and thresholds produce the same queue. The output is a decision-support list for human review.


In [2]:

from pathlib import Path
import pandas as pd
import numpy as np

# Load the anonymized starter dataset.
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("/content/data/raw/content_refresh_anonymized.csv"),
]

csv_path = next((p for p in possible_paths if p.exists()), None)

if csv_path is not None:
    df = pd.read_csv(csv_path)
    print("Loaded local starter dataset.")
else:
    # Public, anonymized starter dataset; no token is needed.
    source_url = (
        "https://raw.githubusercontent.com/"
        "flyrank-bih/flyrank-ml-internship-starter/"
        "main/data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(source_url)
    print("Loaded public anonymized starter dataset.")

required = [
    "content_id",
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Use numeric values carefully. avg_position == 0 means no data.
for col in [
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Comparison group: visible pages with a valid position from 1 to 30.
comparison_mask = (
    df["impressions_90d"].ge(MIN_IMPRESSIONS)
    & df["avg_position"].between(1, STRIKING_MAX_POSITION)
    & df["ctr"].notna()
)

if not comparison_mask.any():
    raise ValueError("No eligible pages to calculate the CTR median.")

ctr_median = df.loc[comparison_mask, "ctr"].median()
print(f"Comparison-group CTR median: {ctr_median:.3f}")
print("This is computed from the dataset, not a preset result.")

# Define the review signals.
df["high_visibility"] = df["impressions_90d"].ge(
    MIN_IMPRESSIONS
)

df["striking_distance"] = df["avg_position"].between(
    STRIKING_MIN_POSITION,
    STRIKING_MAX_POSITION,
)

df["stale_content"] = df["days_since_last_update"].ge(
    STALE_DAYS
)

df["low_ctr"] = (
    df["ctr"].lt(ctr_median)
    & df["avg_position"].between(1, STRIKING_MAX_POSITION)
)

# Only high-visibility pages can receive a positive score.
eligible = df["high_visibility"]

df["baseline_score"] = np.where(
    eligible,
    WEIGHT_STRIKING * df["striking_distance"].astype(int)
    + WEIGHT_STALE * df["stale_content"].astype(int)
    + WEIGHT_LOW_CTR * df["low_ctr"].astype(int),
    0,
)

# Add reason codes.
def get_reason_codes(row):
    if not row["high_visibility"]:
        return "low_visibility"

    reasons = []
    if row["striking_distance"]:
        reasons.append("striking_distance")
    if row["stale_content"]:
        reasons.append("stale_content")
    if row["low_ctr"]:
        reasons.append("low_ctr")

    return ",".join(reasons) if reasons else "no_rule_trigger"

df["reason_codes"] = df.apply(get_reason_codes, axis=1)

# Recommended next action, not an automatic change.
def get_action(row):
    if not row["high_visibility"]:
        return "monitor_or_triage"
    if row["striking_distance"] and row["low_ctr"]:
        return "review_title_and_snippet"
    if row["striking_distance"]:
        return "review_search_intent_and_content"
    if row["stale_content"]:
        return "review_content_freshness"
    if row["low_ctr"]:
        return "review_title_and_snippet"
    return "manual_triage"

df["recommended_action"] = df.apply(get_action, axis=1)

# Confidence is a data-quality note, not a calibrated probability.
def get_confidence_note(row):
    if not row["high_visibility"]:
        return "low: below visibility threshold"
    if row["avg_position"] <= 0 or pd.isna(row["avg_position"]):
        return "limited: search position unavailable"
    n_signals = sum([
        bool(row["striking_distance"]),
        bool(row["stale_content"]),
        bool(row["low_ctr"]),
    ])
    if n_signals >= 2:
        return "moderate: multiple rule signals"
    if n_signals == 1:
        return "limited: one rule signal"
    return "low: no positive rule signal"

df["confidence_note"] = df.apply(get_confidence_note, axis=1)

# Stable tie-breaking by content ID. ID is not part of the score.
ranked = df.sort_values(
    ["baseline_score", "impressions_90d", "content_id"],
    ascending=[False, False, True],
    kind="mergesort",
).copy()

ranked["rank"] = np.arange(1, len(ranked) + 1)

# Write only the review-queue fields, not the entire raw dataset.
output_cols = [
    "rank",
    "content_id",
    "baseline_score",
    "recommended_action",
    "reason_codes",
    "confidence_note",
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
]

output_cols = [c for c in output_cols if c in ranked.columns]

out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

ranked[output_cols].to_csv(out_path, index=False)

print("Rows ranked:", len(ranked))
print("Unique content IDs:", ranked["content_id"].nunique())
print("Output written to:", out_path)
display(ranked[output_cols].head(10))

assert ranked["content_id"].is_unique
assert ranked["rank"].is_monotonic_increasing
assert out_path.exists()

Loaded public anonymized starter dataset.
Comparison-group CTR median: 0.190
This is computed from the dataset, not a preset result.
Rows ranked: 30000
Unique content IDs: 30000
Output written to: work/outputs/baseline_action_score.csv


,rank,content_id,baseline_score,recommended_action,reason_codes,confidence_note,impressions_90d,avg_position,ctr,days_since_last_update
16751,1,content_cf56e2e2e282,6,review_title_and_snippet,"striking_distance,stale_content,low_ctr",moderate: multiple rule signals,61678,19.7,0.15,194
16514,2,content_7368877ea310,6,review_title_and_snippet,"striking_distance,stale_content,low_ctr",moderate: multiple rule signals,59472,24.8,0.13,194
20837,3,content_928af3e22c80,6,review_title_and_snippet,"striking_distance,stale_content,low_ctr",moderate: multiple rule signals,1697,15.8,0.12,193
23215,4,content_bdbec75c1148,6,review_title_and_snippet,"striking_distance,stale_content,low_ctr",moderate: multiple rule signals,1316,21.8,0.15,194
11630,5,content_6226ee6adc91,6,review_title_and_snippet,"striking_distance,stale_content,low_ctr",moderate: multiple rule signals,545,17.8,0.18,183
7021,6,content_1bfaa38ff26c,5,review_search_intent_and_content,"striking_distance,stale_content",moderate: multiple rule signals,25715,22.2,0.23,194
12045,7,content_c2d929d83eaa,5,review_search_intent_and_content,"striking_distance,stale_content",moderate: multiple rule signals,7558,17.9,0.20,193
5327,8,content_fe16a55cd13d,5,review_search_intent_and_content,"striking_distance,stale_content",moderate: multiple rule signals,4556,16.4,0.33,194
26810,9,content_ecb6215e79fd,5,review_search_intent_and_content,"striking_distance,stale_content",moderate: multiple rule signals,4429,25.3,0.38,194
26799,10,content_77d4d5930e5e,5,review_search_intent_and_content,"striking_distance,stale_content",moderate: multiple rule signals,828,18.6,0.24,194


## 3. Top-20 review

The top 20 will be reviewed individually. For each page, I will record its proposed action, the reason code, a confidence note based on the available signals, and what could make the recommendation wrong.

A score is not a probability. Pages with high scores can still be weak picks if their data is incomplete, the comparison group is unsuitable, or the underlying issue is unrelated to the proposed action. The generated table is a review worksheet; I will not treat its suggested explanations as completed human verification.


In [3]:

# Section 3: Create the top-20 human review worksheet.

top20 = ranked.head(20).copy()

def what_could_make_it_wrong(row):
    risks = []

    if pd.isna(row["avg_position"]) or row["avg_position"] <= 0:
        risks.append("No valid average search position")
    if pd.isna(row["ctr"]):
        risks.append("CTR missing")
    if row["impressions_90d"] < MIN_IMPRESSIONS:
        risks.append("Low impression volume")
    if row["days_since_last_update"] < STALE_DAYS:
        risks.append("Staleness signal not present")
    if not row["striking_distance"]:
        risks.append("Striking-distance signal not present")
    if not row["low_ctr"]:
        risks.append("Low-CTR signal not present")

    if not risks:
        risks.append(
            "Check search intent, seasonality, technical issues, "
            "and whether the proposed action fits the page"
        )

    return "; ".join(risks)

top20["what_could_make_it_wrong"] = top20.apply(
    what_could_make_it_wrong, axis=1
)

top20["manual_review_status"] = "pending human review"

review_cols = [
    "rank",
    "content_id",
    "baseline_score",
    "recommended_action",
    "reason_codes",
    "confidence_note",
    "what_could_make_it_wrong",
    "manual_review_status",
]

display(top20[review_cols])

# Save the worksheet separately so it can be reviewed and annotated.
review_path = Path("work/outputs/top20_review.csv")
review_path.parent.mkdir(parents=True, exist_ok=True)
top20[review_cols].to_csv(review_path, index=False)

print("Review worksheet saved to:", review_path)
print("Complete the manual review status after inspecting the picks.")

,rank,content_id,baseline_score,recommended_action,reason_codes,confidence_note,what_could_make_it_wrong,manual_review_status
16751,1,content_cf56e2e2e282,6,review_title_and_snippet,"striking_distance,stale_content,low_ctr",moderate: multiple rule signals,"Check search intent, seasonality, technical is...",pending human review
16514,2,content_7368877ea310,6,review_title_and_snippet,"striking_distance,stale_content,low_ctr",moderate: multiple rule signals,"Check search intent, seasonality, technical is...",pending human review
20837,3,content_928af3e22c80,6,review_title_and_snippet,"striking_distance,stale_content,low_ctr",moderate: multiple rule signals,"Check search intent, seasonality, technical is...",pending human review
23215,4,content_bdbec75c1148,6,review_title_and_snippet,"striking_distance,stale_content,low_ctr",moderate: multiple rule signals,"Check search intent, seasonality, technical is...",pending human review
11630,5,content_6226ee6adc91,6,review_title_and_snippet,"striking_distance,stale_content,low_ctr",moderate: multiple rule signals,"Check search intent, seasonality, technical is...",pending human review
7021,6,content_1bfaa38ff26c,5,review_search_intent_and_content,"striking_distance,stale_content",moderate: multiple rule signals,Low-CTR signal not present,pending human review
12045,7,content_c2d929d83eaa,5,review_search_intent_and_content,"striking_distance,stale_content",moderate: multiple rule signals,Low-CTR signal not present,pending human review
5327,8,content_fe16a55cd13d,5,review_search_intent_and_content,"striking_distance,stale_content",moderate: multiple rule signals,Low-CTR signal not present,pending human review
26810,9,content_ecb6215e79fd,5,review_search_intent_and_content,"striking_distance,stale_content",moderate: multiple rule signals,Low-CTR signal not present,pending human review
26799,10,content_77d4d5930e5e,5,review_search_intent_and_content,"striking_distance,stale_content",moderate: multiple rule signals,Low-CTR signal not present,pending human review


Review worksheet saved to: work/outputs/top20_review.csv
Complete the manual review status after inspecting the picks.


## 4. Weak picks + leakage check

Some high-ranked pages may be weak picks because the rule uses coarse thresholds and does not understand search intent, page quality, seasonality, technical errors, or business value.

I will inspect the top 20 for missing or invalid measurements and check that no label-derived trend fields, existing product-decision flags, identifiers, or future-window fields were used to calculate the score.

This baseline ranks using current snapshot signals only. It does not establish that the recommended action will improve traffic.


In [4]:

# Identify potentially weak picks among the top 20.
weak_pick_mask = (
    top20["avg_position"].isna()
    | top20["avg_position"].le(0)
    | top20["ctr"].isna()
    | top20["reason_codes"].eq("no_rule_trigger")
)

weak_picks = top20.loc[weak_pick_mask, review_cols]

print("Potentially weak picks in top 20:", len(weak_picks))
display(weak_picks)

# Leakage audit: these fields are forbidden as score inputs.
forbidden_exact = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "measurable_opportunity",
    "needs_ctr_fix",
    "quick_win",
    "action_type",
    "priority_score",
}

# Explicitly document the score inputs.
score_inputs = {
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
}

leakage_hits = score_inputs & forbidden_exact

assert not leakage_hits, (
    f"Forbidden fields found in score inputs: {leakage_hits}"
)

assert score_inputs.issubset(set(df.columns)), (
    "At least one score input is missing from the dataframe."
)

print("Score inputs:", sorted(score_inputs))
print("Forbidden-field overlap:", leakage_hits)
print("PASS: no listed label, identifier, or product flag is a score input.")

# Optional proxy evaluation, kept separate from score construction.
# This label is used only for evaluation, never to rank the pages.
if "trend_direction" in df.columns:
    evaluation_labels = (
        ranked["trend_direction"].eq("down").astype(int)
    )
    base_rate = evaluation_labels.mean()

    print(f"\nDecline-proxy base rate: {base_rate:.3f}")

    for k in [20, 50]:
        if len(ranked) >= k:
            precision_k = evaluation_labels.head(k).mean()
            print(
                f"Precision@{k}: {precision_k:.3f} "
                f"(proxy-label base rate: {base_rate:.3f})"
            )
else:
    print(
        "No trend_direction field available; skip proxy evaluation "
        "until the label is loaded separately."
    )

Potentially weak picks in top 20: 0


,rank,content_id,baseline_score,recommended_action,reason_codes,confidence_note,what_could_make_it_wrong,manual_review_status


Score inputs: ['avg_position', 'ctr', 'days_since_last_update', 'impressions_90d']
Forbidden-field overlap: set()
PASS: no listed label, identifier, or product flag is a score input.

Decline-proxy base rate: 0.542
Precision@20: 0.700 (proxy-label base rate: 0.542)
Precision@50: 0.500 (proxy-label base rate: 0.542)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.